In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pickle

In [ ]:
# Load the dataset
data_df = pd.read_csv('Churn_Modelling.csv')
data_df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


Preprocess the data

In [ ]:
# Drop unnecessary columns
data_df = data_df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [22]:
# Encode categorical variables
le_gender = LabelEncoder()
data_df['Gender'] = le_gender.fit_transform(data_df['Gender'])
data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [ ]:
# One-hot encode 'Geography' column
# This is being done to since 'Geography' has more than two categories

from sklearn.preprocessing import OneHotEncoder
ohe_geography = OneHotEncoder(drop='first', sparse_output=False)
geography_encoded = ohe_geography.fit_transform(data_df[['Geography']])
ohe_geography.get_feature_names_out()



array(['Geography_Germany', 'Geography_Spain'], dtype=object)

In [29]:
geo_encoded_df = pd.DataFrame(geography_encoded, columns=ohe_geography.get_feature_names_out())
geo_encoded_df

,Geography_Germany,Geography_Spain
0,0.0,0.0
1,0.0,1.0
2,0.0,0.0
3,0.0,0.0
4,0.0,1.0
...,...,...
9995,0.0,0.0
9996,0.0,0.0
9997,0.0,0.0
9998,1.0,0.0


In [30]:
# Combine the encoded geography columns back to the main dataframe
data_df = pd.concat([data_df.drop('Geography', axis=1), geo_encoded_df], axis=1)
data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,1.0,0.0


In [31]:
# Save the encoders for future use
with open('le_gender.pkl', 'wb') as f:
    pickle.dump(le_gender, f)

with open('ohe_geography.pkl', 'wb') as f:
    pickle.dump(ohe_geography, f)



In [32]:
# Divide the dataset into features and target variable
X = data_df.drop('Exited', axis=1)
y = data_df['Exited']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



In [34]:
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

Artificial Neural Network (ANN) Implementation

In [35]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [39]:
# Build the ANN model
model = Sequential(Dense(units=64, activation='relu', input_shape=(X_train.shape[1],))) ## Hidden Layer 1 Connected with input layer
model.add(Dense(units=32, activation='relu'))  ## Hidden Layer 2
model.add(Dense(units=1, activation='sigmoid'))  ## Output Layer

In [40]:
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_2 (Dense)             (None, 64)                768       
                                                                 
 dense_3 (Dense)             (None, 32)                2080      
                                                                 
 dense_4 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2881 (11.25 KB)
Trainable params: 2881 (11.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [41]:
opt = tf.keras.optimizers.Adam(learning_rate=0.001)
loss = tf.keras.losses.BinaryCrossentropy()

# Compile the model
model.compile(optimizer=opt, loss=loss, metrics=['accuracy'])

In [42]:
# Set up the TensorBoard callback
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%  H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [44]:
# Setup Early Stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


In [45]:
# Train the model
history = model.fit(X_train, y_train, validation_split=0.2, epochs=100, batch_size=32,
                    callbacks=[early_stopping, tensorboard_callback])

# Evaluate the model on the test set
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {test_loss}, Test Accuracy: {test_accuracy}')

Epoch 1/100
200/200 [==============================] - 0s 973us/step - loss: 0.4576 - accuracy: 0.8008 - val_loss: 0.3987 - val_accuracy: 0.8406
Epoch 2/100
200/200 [==============================] - 0s 968us/step - loss: 0.3913 - accuracy: 0.8388 - val_loss: 0.3674 - val_accuracy: 0.8512
Epoch 3/100
200/200 [==============================] - 0s 490us/step - loss: 0.3604 - accuracy: 0.8522 - val_loss: 0.3557 - val_accuracy: 0.8481
Epoch 4/100
200/200 [==============================] - 0s 487us/step - loss: 0.3501 - accuracy: 0.8566 - val_loss: 0.3522 - val_accuracy: 0.8512
Epoch 5/100
200/200 [==============================] - 0s 482us/step - loss: 0.3439 - accuracy: 0.8597 - val_loss: 0.3480 - val_accuracy: 0.8562
Epoch 6/100
200/200 [==============================] - 0s 478us/step - loss: 0.3402 - accuracy: 0.8603 - val_loss: 0.3500 - val_accuracy: 0.8544
Epoch 7/100
200/200 [==============================] - 0s 488us/step - loss: 0.3364 - accuracy: 0.8625 - val_loss: 0.3483 - val_ac

In [46]:
# Save the model
model.save('ann_churn_model.h5')

/Users/satyakibasu/Documents/Satyaki/python_code/gen-ai/p311env/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [48]:
# Load TensorBoard in Jupyter Notebook
%load_ext tensorboard
%tensorboard --logdir logs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 38508), started 0:01:23 ago. (Use '!kill 38508' to kill it.)